# 05 — Features

**Étape 7 — Features (lags, rolling, calendrier, cycliques, fondamentaux)**

Construire le pipeline de feature engineering réutilisable (`src/features.py`),
en respectant strictement les règles anti-leakage (section 11.3).


## Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path(
    "/Users/evitafaj/Desktop/Portofolio/"
    "electricity-price-forecasting"
)

DATASET_PATH = (
    REPO_ROOT / "data" / "processed" / "dataset_v1.csv"
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import features

pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (14, 5)

dataset = pd.read_csv(
    DATASET_PATH,
    parse_dates=["datetime"],
    index_col="datetime",
)

dataset.index = pd.to_datetime(
    dataset.index,
    utc=True,
)

dataset = dataset.sort_index()

print("Dimensions :", dataset.shape)
print("Début :", dataset.index.min())
print("Fin :", dataset.index.max())
print("Doublons :", dataset.index.duplicated().sum())
print("Valeurs manquantes :", dataset.isna().sum().sum())

dataset.head()

Dimensions : (32855, 17)
Début : 2021-12-31 23:00:00+00:00
Fin : 2025-09-30 21:00:00+00:00
Doublons : 0
Valeurs manquantes : 0


,price_da,temperature_2m,wind_speed_10m,shortwave_radiation,demand_actual,demand_forecast_j1,oil_generation,coal_generation,gas_generation,nuclear_generation,wind_generation,solar_generation,hydro_generation,pumping_consumption,bioenergy_generation,net_import_mw,fundamentals_imputed
datetime,,,,,,,,,,,,,,,,,
2021-12-31 23:00:00+00:00,89.06,9.4,3.111111,0.0,54315.0,50787.5,94.0,13.5,2826.0,40497.0,3187.5,0.0,9465.5,-264.0,1247.5,-2679.0,0
2022-01-01 00:00:00+00:00,78.48,10.7,2.500000,0.0,52173.5,49162.5,99.0,13.5,2693.0,38645.5,3270.0,0.0,9108.0,-65.0,1255.0,-2768.5,0
2022-01-01 01:00:00+00:00,85.16,9.3,2.638889,0.0,51527.5,47550.0,99.0,13.0,2687.0,38081.0,3146.0,0.0,8427.0,-602.5,1250.5,-1491.0,0
2022-01-01 02:00:00+00:00,50.00,8.8,2.833333,0.0,48845.5,44687.5,99.0,13.0,2694.0,37259.5,3273.5,0.0,6787.0,-1459.5,1247.5,-986.5,0
2022-01-01 03:00:00+00:00,37.67,8.2,2.944444,0.0,46646.0,43075.0,99.0,13.0,2699.5,36374.5,3481.5,0.0,6443.0,-2638.0,1253.5,-1001.0,0


## Lags calendaires robustes aux changements d'heure : J-1, J-2, J-7 (`add_price_day_lags`)

In [2]:
feature_df = features.add_price_day_lags(
    dataset,
    price_col="price_da",
    day_lags=(1, 2, 7),
    timezone="Europe/Paris",
)

lag_columns = [
    "price_da_lag_1d",
    "price_da_lag_2d",
    "price_da_lag_7d",
]

print("Valeurs disponibles :")
print(feature_df[lag_columns].notna().sum())

print("\nValeurs manquantes initiales :")
print(feature_df[lag_columns].isna().sum())

display(
    feature_df[
        ["price_da"] + lag_columns
    ].head(175)
)

Valeurs disponibles :
price_da_lag_1d    32824
price_da_lag_2d    32800
price_da_lag_7d    32680
dtype: int64

Valeurs manquantes initiales :
price_da_lag_1d     31
price_da_lag_2d     55
price_da_lag_7d    175
dtype: int64


,price_da,price_da_lag_1d,price_da_lag_2d,price_da_lag_7d
datetime,,,,
2021-12-31 23:00:00+00:00,89.06,NaN,NaN,NaN
2022-01-01 00:00:00+00:00,78.48,NaN,NaN,NaN
2022-01-01 01:00:00+00:00,85.16,NaN,NaN,NaN
2022-01-01 02:00:00+00:00,50.00,NaN,NaN,NaN
2022-01-01 03:00:00+00:00,37.67,NaN,NaN,NaN
...,...,...,...,...
2022-01-08 01:00:00+00:00,166.00,180.0,191.82,85.16
2022-01-08 02:00:00+00:00,161.63,164.9,180.70,50.00
2022-01-08 03:00:00+00:00,160.00,162.8,170.10,37.67


## Rolling mean/std sur les jours locaux antérieurs (`add_price_daily_rolling_stats`)

In [3]:
feature_df = features.add_price_daily_rolling_stats(
    feature_df,
    price_col="price_da",
    windows_days=(1, 7),
    timezone="Europe/Paris",
)

rolling_columns = [
    "price_da_rollmean_1d",
    "price_da_rollstd_1d",
    "price_da_rollmean_7d",
    "price_da_rollstd_7d",
]

print("Valeurs manquantes initiales :")
print(feature_df[rolling_columns].isna().sum())

display(
    feature_df[
        ["price_da"] + rolling_columns
    ].head(200)
)

Valeurs manquantes initiales :
price_da_rollmean_1d     24
price_da_rollstd_1d      24
price_da_rollmean_7d    168
price_da_rollstd_7d     168
dtype: int64


,price_da,price_da_rollmean_1d,price_da_rollstd_1d,price_da_rollmean_7d,price_da_rollstd_7d
datetime,,,,,
2021-12-31 23:00:00+00:00,89.06,NaN,NaN,NaN,NaN
2022-01-01 00:00:00+00:00,78.48,NaN,NaN,NaN,NaN
2022-01-01 01:00:00+00:00,85.16,NaN,NaN,NaN,NaN
2022-01-01 02:00:00+00:00,50.00,NaN,NaN,NaN,NaN
2022-01-01 03:00:00+00:00,37.67,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2022-01-09 02:00:00+00:00,55.46,176.557083,28.424627,157.333155,71.899807
2022-01-09 03:00:00+00:00,45.82,176.557083,28.424627,157.333155,71.899807
2022-01-09 04:00:00+00:00,48.50,176.557083,28.424627,157.333155,71.899807


## Features calendaires + encodage cyclique heure/jour (`add_calendar_features`)

In [4]:
feature_df = features.add_calendar_features(
    feature_df,
    timezone="Europe/Paris",
)

# add_calendar_features crée déjà :
# hour, weekday, is_weekend, month,
# hour_sin, hour_cos, weekday_sin, weekday_cos

feature_df["month_sin"] = np.sin(
    2 * np.pi * feature_df["month"] / 12
)

feature_df["month_cos"] = np.cos(
    2 * np.pi * feature_df["month"] / 12
)

calendar_columns = [
    "hour",
    "weekday",
    "is_weekend",
    "month",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
]

display(feature_df[calendar_columns].head(30))

print(
    "Valeurs manquantes :",
    feature_df[calendar_columns].isna().sum().sum(),
)

,hour,weekday,is_weekend,month,hour_sin,hour_cos,weekday_sin,weekday_cos,month_sin,month_cos
datetime,,,,,,,,,,
2021-12-31 23:00:00+00:00,0,5,1,1,0.000000e+00,1.000000e+00,-0.974928,-0.222521,0.5,0.866025
2022-01-01 00:00:00+00:00,1,5,1,1,2.588190e-01,9.659258e-01,-0.974928,-0.222521,0.5,0.866025
2022-01-01 01:00:00+00:00,2,5,1,1,5.000000e-01,8.660254e-01,-0.974928,-0.222521,0.5,0.866025
2022-01-01 02:00:00+00:00,3,5,1,1,7.071068e-01,7.071068e-01,-0.974928,-0.222521,0.5,0.866025
2022-01-01 03:00:00+00:00,4,5,1,1,8.660254e-01,5.000000e-01,-0.974928,-0.222521,0.5,0.866025
2022-01-01 04:00:00+00:00,5,5,1,1,9.659258e-01,2.588190e-01,-0.974928,-0.222521,0.5,0.866025
2022-01-01 05:00:00+00:00,6,5,1,1,1.000000e+00,6.123234e-17,-0.974928,-0.222521,0.5,0.866025
2022-01-01 06:00:00+00:00,7,5,1,1,9.659258e-01,-2.588190e-01,-0.974928,-0.222521,0.5,0.866025
2022-01-01 07:00:00+00:00,8,5,1,1,8.660254e-01,-5.000000e-01,-0.974928,-0.222521,0.5,0.866025


Valeurs manquantes : 0


## Jours fériés français (package `holidays`)

In [5]:
import holidays

local_dates = pd.Series(
    feature_df.index
    .tz_convert("Europe/Paris")
    .date,
    index=feature_df.index,
)

years = sorted(
    feature_df.index
    .tz_convert("Europe/Paris")
    .year
    .unique()
)

fr_holidays = holidays.France(
    years=years,
)

feature_df["is_holiday"] = (
    local_dates.isin(fr_holidays)
    .astype(int)
)

print("Années :", years)
print(
    "Nombre d'heures fériées :",
    feature_df["is_holiday"].sum(),
)

holiday_examples = feature_df.loc[
    feature_df["is_holiday"].eq(1),
    ["price_da", "is_holiday"],
]

display(holiday_examples.head(48))

Années : [2022, 2023, 2024, 2025]
Nombre d'heures fériées : 984


,price_da,is_holiday
datetime,,
2021-12-31 23:00:00+00:00,89.06,1
2022-01-01 00:00:00+00:00,78.48,1
2022-01-01 01:00:00+00:00,85.16,1
2022-01-01 02:00:00+00:00,50.00,1
2022-01-01 03:00:00+00:00,37.67,1
2022-01-01 04:00:00+00:00,39.70,1
2022-01-01 05:00:00+00:00,40.59,1
2022-01-01 06:00:00+00:00,43.26,1
2022-01-01 07:00:00+00:00,49.66,1


## Residual load et part de renouvelables (`add_residual_load`)

In [15]:
feature_df["residual_load_actual_diagnostic"] = (
    feature_df["demand_actual"]
    - feature_df["wind_generation"]
    - feature_df["solar_generation"]
)

feature_df["variable_renewables_share_actual_diagnostic"] = (
    feature_df["wind_generation"]
    + feature_df["solar_generation"]
) / feature_df["demand_actual"]

feature_df[
    [
        "demand_actual",
        "wind_generation",
        "solar_generation",
        "residual_load_actual_diagnostic",
        "variable_renewables_share_actual_diagnostic",
    ]
].head()

,demand_actual,wind_generation,solar_generation,residual_load_actual_diagnostic,variable_renewables_share_actual_diagnostic
datetime,,,,,
2021-12-31 23:00:00+00:00,54315.0,3187.5,0.0,51127.5,0.058685
2022-01-01 00:00:00+00:00,52173.5,3270.0,0.0,48903.5,0.062675
2022-01-01 01:00:00+00:00,51527.5,3146.0,0.0,48381.5,0.061055
2022-01-01 02:00:00+00:00,48845.5,3273.5,0.0,45572.0,0.067017
2022-01-01 03:00:00+00:00,46646.0,3481.5,0.0,43164.5,0.074637


## Interactions (residual load × gas, low nuclear × demand)

In [18]:
# Période d'entraînement :
# du début des données au 1er janvier 2024, heure française
local_index = feature_df.index.tz_convert("Europe/Paris")

train_mask = local_index < pd.Timestamp(
    "2024-01-01",
    tz="Europe/Paris",
)

# Le seuil est appris uniquement sur le train
low_nuclear_threshold = feature_df.loc[
    train_mask,
    "nuclear_generation",
].quantile(0.10)

feature_df["low_nuclear_actual_diagnostic"] = (
    feature_df["nuclear_generation"] <= low_nuclear_threshold
).astype("int8")

feature_df["residual_load_x_gas_actual_diagnostic"] = (
    feature_df["residual_load_actual_diagnostic"] / 1000
) * (
    feature_df["gas_generation"] / 1000
)

feature_df["low_nuclear_x_demand_actual_diagnostic"] = (
    feature_df["low_nuclear_actual_diagnostic"]
    * feature_df["demand_actual"] / 1000
)

print(
    "Nombre d'heures dans le train :",
    train_mask.sum(),
)

print(
    "Seuil bas nucléaire appris sur le train :",
    round(low_nuclear_threshold / 1000, 2),
    "GW",
)

feature_df[
    [
        "low_nuclear_actual_diagnostic",
        "residual_load_x_gas_actual_diagnostic",
        "low_nuclear_x_demand_actual_diagnostic",
    ]
].head()

Nombre d'heures dans le train : 17520
Seuil bas nucléaire appris sur le train : 25.63 GW


,low_nuclear_actual_diagnostic,residual_load_x_gas_actual_diagnostic,low_nuclear_x_demand_actual_diagnostic
datetime,,,
2021-12-31 23:00:00+00:00,0,144.486315,0.0
2022-01-01 00:00:00+00:00,0,131.697125,0.0
2022-01-01 01:00:00+00:00,0,130.001091,0.0
2022-01-01 02:00:00+00:00,0,122.770968,0.0
2022-01-01 03:00:00+00:00,0,116.522568,0.0


## Étiquettes de régime pour l'analyse uniquement (`fit_spike_threshold`, `add_regime_flags`)

In [32]:
spike_threshold_q95 = feature_df.loc[
    train_mask,
    "price_da",
].quantile(0.95)

feature_df["negative_price_flag_analysis_only"] = (
    feature_df["price_da"] < 0
).astype("int8")

feature_df["spike_flag_analysis_only"] = (
    feature_df["price_da"] > spike_threshold_q95
).astype("int8")

print(
    "Seuil de spike Q95 appris sur le train :",
    round(spike_threshold_q95, 2),
    "EUR/MWh",
)

print(
    feature_df[
        [
            "negative_price_flag_analysis_only",
            "spike_flag_analysis_only",
        ]
    ].sum()
)

Seuil de spike Q95 appris sur le train : 472.38 EUR/MWh
negative_price_flag_analysis_only    996
spike_flag_analysis_only             877
dtype: int64


In [34]:
SAFE_MODEL_FEATURES = [
    # Prix historiques déjà disponibles
    "price_da_lag_1d",
    "price_da_lag_2d",
    "price_da_lag_7d",
    "price_da_rollmean_1d",
    "price_da_rollstd_1d",
    "price_da_rollmean_7d",
    "price_da_rollstd_7d",

    # Calendrier connu à l'avance
    "hour",
    "weekday",
    "is_weekend",
    "month",
    "hour_sin",
    "hour_cos",
    "weekday_sin",
    "weekday_cos",
    "month_sin",
    "month_cos",
    "is_holiday",
]

CANDIDATE_AFTER_PUBLICATION_CHECK = [
    "demand_forecast_j1",
]

DIAGNOSTIC_ONLY_FEATURES = [
    "temperature_2m",
    "wind_speed_10m",
    "shortwave_radiation",
    "demand_actual",
    "oil_generation",
    "coal_generation",
    "gas_generation",
    "nuclear_generation",
    "wind_generation",
    "solar_generation",
    "hydro_generation",
    "pumping_consumption",
    "bioenergy_generation",
    "net_import_mw",
    "residual_load_actual_diagnostic",
    "variable_renewables_share_actual_diagnostic",
    "low_nuclear_actual_diagnostic",
    "residual_load_x_gas_actual_diagnostic",
    "low_nuclear_x_demand_actual_diagnostic",
]

ANALYSIS_ONLY_LABELS = [
    "negative_price_flag_analysis_only",
    "spike_flag_analysis_only",
]

print(
    "Features opérationnelles sûres :",
    len(SAFE_MODEL_FEATURES),
)

print(
    "Candidates à vérifier :",
    CANDIDATE_AFTER_PUBLICATION_CHECK,
)

print(
    "Variables diagnostic_only :",
    len(DIAGNOSTIC_ONLY_FEATURES),
)

Features opérationnelles sûres : 18
Candidates à vérifier : ['demand_forecast_j1']
Variables diagnostic_only : 19


## Vérification anti-leakage : aucune feature ne doit utiliser d'info future

In [35]:
forbidden_model_columns = (
    ["price_da"]
    + DIAGNOSTIC_ONLY_FEATURES
    + ANALYSIS_ONLY_LABELS
)

assert not (
    set(SAFE_MODEL_FEATURES)
    & set(forbidden_model_columns)
)

assert set(SAFE_MODEL_FEATURES).issubset(
    feature_df.columns
)

assert set(ANALYSIS_ONLY_LABELS).issubset(
    feature_df.columns
)

assert feature_df.index.is_monotonic_increasing
assert feature_df.index.duplicated().sum() == 0

print(
    "Aucune cible, variable réalisée ou étiquette "
    "de régime dans SAFE_MODEL_FEATURES."
)

Aucune cible, variable réalisée ou étiquette de régime dans SAFE_MODEL_FEATURES.


In [36]:
temporal_feature_columns = [
    "price_da_lag_1d",
    "price_da_lag_2d",
    "price_da_lag_7d",
    "price_da_rollmean_1d",
    "price_da_rollstd_1d",
    "price_da_rollmean_7d",
    "price_da_rollstd_7d",
]

counterfactual = dataset.copy()

counterfactual.loc[
    counterfactual.index >= validation_start,
    "price_da",
] = 1_000_000.0

counterfactual = features.add_price_day_lags(
    counterfactual,
    price_col="price_da",
    day_lags=(1, 2, 7),
    timezone="Europe/Paris",
)

counterfactual = (
    features.add_price_daily_rolling_stats(
        counterfactual,
        price_col="price_da",
        windows_days=(1, 7),
        timezone="Europe/Paris",
    )
)

pd.testing.assert_frame_equal(
    feature_df.loc[
        feature_df.index < validation_start,
        temporal_feature_columns,
    ],
    counterfactual.loc[
        counterfactual.index < validation_start,
        temporal_feature_columns,
    ],
)

print(
    "Test anti-futur validé : modifier les prix "
    "futurs ne change pas les features historiques."
)

Test anti-futur validé : modifier les prix futurs ne change pas les features historiques.


In [37]:
missing_safe = (
    feature_df[SAFE_MODEL_FEATURES]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Valeurs manquantes par feature sûre :")
display(missing_safe.to_frame("missing"))

complete_safe_mask = (
    feature_df[SAFE_MODEL_FEATURES]
    .notna()
    .all(axis=1)
)

print(
    "Lignes complètes pour la modélisation :",
    complete_safe_mask.sum(),
)

print(
    "Première ligne complète :",
    feature_df.index[complete_safe_mask][0],
)

Valeurs manquantes par feature sûre :


,missing
price_da_lag_7d,175
price_da_rollmean_7d,168
price_da_rollstd_7d,168
price_da_lag_2d,55
price_da_lag_1d,31
price_da_rollmean_1d,24
price_da_rollstd_1d,24
hour_cos,0
month_cos,0
month_sin,0


Lignes complètes pour la modélisation : 32672
Première ligne complète : 2022-01-07 23:00:00+00:00


## Export du dataset avec features vers `data/processed/dataset_features.csv`

In [38]:
# Vérifier les éventuels noms de colonnes dupliqués
duplicated_columns = feature_df.columns[
    feature_df.columns.duplicated(keep=False)
].tolist()

print("Colonnes dupliquées avant nettoyage :", duplicated_columns)

# Conserver une seule occurrence de chaque colonne
feature_df = feature_df.loc[
    :,
    ~feature_df.columns.duplicated(keep="last"),
].copy()

# Éviter également les doublons dans les listes du manifeste
SAFE_MODEL_FEATURES = list(dict.fromkeys(SAFE_MODEL_FEATURES))
CANDIDATE_AFTER_PUBLICATION_CHECK = list(
    dict.fromkeys(CANDIDATE_AFTER_PUBLICATION_CHECK)
)
DIAGNOSTIC_ONLY_FEATURES = list(
    dict.fromkeys(DIAGNOSTIC_ONLY_FEATURES)
)
ANALYSIS_ONLY_LABELS = list(
    dict.fromkeys(ANALYSIS_ONLY_LABELS)
)

assert feature_df.columns.is_unique
assert len(ANALYSIS_ONLY_LABELS) == 2

print("Étiquettes d’analyse :", ANALYSIS_ONLY_LABELS)
print(
    feature_df[
        [
            "negative_price_flag_analysis_only",
            "spike_flag_analysis_only",
        ]
    ].sum()
)

Colonnes dupliquées avant nettoyage : []
Étiquettes d’analyse : ['negative_price_flag_analysis_only', 'spike_flag_analysis_only']
negative_price_flag_analysis_only    996
spike_flag_analysis_only             877
dtype: int64


In [39]:
PROCESSED_DIR = (
    REPO_ROOT / "data" / "processed"
)

FEATURE_DATASET_PATH = (
    PROCESSED_DIR / "dataset_features.csv"
)

FEATURE_MANIFEST_PATH = (
    PROCESSED_DIR / "feature_manifest.csv"
)

feature_df.to_csv(
    FEATURE_DATASET_PATH
)

manifest_rows = []

for column in SAFE_MODEL_FEATURES:
    manifest_rows.append(
        {
            "feature": column,
            "status": "model_safe",
        }
    )

for column in CANDIDATE_AFTER_PUBLICATION_CHECK:
    manifest_rows.append(
        {
            "feature": column,
            "status": "candidate_publication_check",
        }
    )

for column in DIAGNOSTIC_ONLY_FEATURES:
    manifest_rows.append(
        {
            "feature": column,
            "status": "diagnostic_only",
        }
    )

for column in ANALYSIS_ONLY_LABELS:
    manifest_rows.append(
        {
            "feature": column,
            "status": "analysis_only_label",
        }
    )

feature_manifest = pd.DataFrame(
    manifest_rows
)

feature_manifest.to_csv(
    FEATURE_MANIFEST_PATH,
    index=False,
)

print(
    "Dataset avec features :",
    FEATURE_DATASET_PATH,
)

print(
    "Dimensions :",
    feature_df.shape,
)

print(
    "Manifest des features :",
    FEATURE_MANIFEST_PATH,
)

Dataset avec features : /Users/evitafaj/Desktop/Portofolio/electricity-price-forecasting/data/processed/dataset_features.csv
Dimensions : (32855, 42)
Manifest des features : /Users/evitafaj/Desktop/Portofolio/electricity-price-forecasting/data/processed/feature_manifest.csv
